# Prediction performance for test data
I try different combinations of marker genes and reference sample set for prediction of age-unknown samples.

## Load Full Training Data
As usual, I load all the age-known samples

In [ ]:
import pandas as pd
import numpy as np
from utils.misc import extract_number, mae
from utils.deconvolution import predict_l2
from utils.viz import single_line_plot, single_scatter_plot
from sklearn.preprocessing import StandardScaler
import os
import matplotlib.pyplot as plt

In [ ]:
gene_expressions = pd.read_csv("data/train_data.csv", index_col=0)
gene_expressions_mat = gene_expressions.to_numpy()
genenames = np.array(gene_expressions.index.tolist())
samples = gene_expressions.columns.tolist()

# extract ages
ages = np.array([extract_number(timestring) for timestring in samples])
unique_ages=np.unique(ages)

# retain genes that are present in all samples
prevalence = np.mean(gene_expressions_mat > 0, axis=1)
subset_gene_id = np.where(prevalence == 1)[0]
subset_genenames = genenames[subset_gene_id]
gene_expressions = gene_expressions.loc[subset_genenames, :]
gene_expressions_mat = gene_expressions_mat[subset_gene_id, :]

# get log expressions
log_gene_expressions = np.log(gene_expressions)
log_gene_expressions_mat = np.log(gene_expressions_mat)

# transpose count tables to samples by genes
gene_expressions = gene_expressions.T
gene_expressions_mat = gene_expressions_mat.T
log_gene_expressions = log_gene_expressions.T
log_gene_expressions_mat = log_gene_expressions_mat.T

# get rankings of samples for each gene expression
# gene_expressions_rank = log_gene_expressions.rank()
gene_expressions_test = pd.read_csv("data/test_data.csv", index_col=0)
gene_expressions_test = gene_expressions_test.T

Let me prepare some utility functions. The first is subsetting the train and test data to contain subset of training samples and subset of marker genes.

In [ ]:
def subset_data(train_expressions, train_ages, test_expressions, marker_genes, min_age=2, max_age=23):

    mask = np.logical_and(train_ages >= min_age, train_ages <= max_age)
    train_ages_subset = train_ages[mask]
    train_expressions_subset = train_expressions.loc[mask, marker_genes]
    # log_train_expressions_subset = np.log(train_expressions_subset)
    test_expressions_subset = test_expressions.loc[:, marker_genes]

    return train_ages_subset, train_expressions_subset, test_expressions_subset


The second carries out deconvolution.

In [ ]:
def predict_test(train_expressions, train_ages, test_expressions):

    train_samples = train_expressions.index.tolist()
    train_expressions = train_expressions.to_numpy()

    test_samples = test_expressions.index.tolist()
    test_expressions = test_expressions.to_numpy()

    predicted_ages = np.zeros(len(test_samples))
    weights = np.zeros((len(test_samples), len(train_samples)))

    log_train_expressions = np.log(train_expressions)
    scaler = StandardScaler()

    scaler.fit(log_train_expressions)
    log_train_expressions_scaled = scaler.transform(log_train_expressions)

    n_test = len(test_samples)
    for j in range(n_test):

        target_expressions = test_expressions[j, :]
        if np.any(target_expressions == 0):
            locations = np.where(target_expressions == 0)[0]
            target_expressions[locations] = np.min(train_expressions[:, locations], axis=0)/2

        log_target_expressions = np.log(target_expressions)
        # standardize
        log_target_expressions_scaled = scaler.transform(log_target_expressions.reshape(1, -1)).flatten()

        estimated_weights, predicted_age = predict_l2(X=log_train_expressions_scaled,
                                                      Y=log_target_expressions_scaled,
                                                      labels=train_ages)

        weights[j, :] = estimated_weights
        predicted_ages[j] = predicted_age

    test_prediction_df = pd.DataFrame({"Sample": test_samples,
                                   "Predicted":predicted_ages})

    weights_test_df = pd.DataFrame(weights, index=test_samples,
                               columns=train_samples)

    return test_prediction_df, weights_test_df


The third function plot and store the estimated deconvlution weights for each sample.

In [ ]:
def save_weights_plot(test_predictions, test_prediction_weights, train_ages, folder):

    if not os.path.exists(folder):
        os.makedirs(folder)

    n_test = len(test_prediction_weights)

    for j in range(n_test):

        selected_weights = test_prediction_weights.iloc[j, :].to_numpy()
        predicted_age = test_predictions["Predicted"][j]
        sample_name = test_prediction_weights.index[j]
        unique_ages = np.unique(train_ages)
        xticks = np.zeros(len(unique_ages))

        # only have one ticks for each unique age
        for index, unique_age in enumerate(unique_ages):
            ids = np.where(train_ages == unique_age)[0]
            xticks[index] = np.min(ids)+1

        xnames = unique_ages.astype(str)

        title = f"{sample_name}|Predicted Age: {predicted_age:.2f}"
        fig = single_line_plot(ymat=selected_weights.reshape(1, -1),
                               xmat=np.arange(1, len(train_ages)+1).reshape(1, -1),
                           xticks=xticks, xticknames=xnames, xname="Age (Months)",
                           yname="Weight",
                           title=title, size=(7, 4))

        filename = f"{sample_name}.pdf"

        fig.savefig(os.path.join(folder, filename), bbox_inches="tight")

        plt.close(fig)


## Reference panel 2-23, Marker gene 2-23

In [ ]:
# predictor genes
predictor_genes_df = pd.read_csv("gene_plots/top_gene_2_23/markergene_2_23.csv", index_col=0)
predictor_genes = predictor_genes_df.index.tolist()

In [ ]:
train_ages, train_expressions, test_expressions = subset_data(train_expressions=gene_expressions,
                                                              train_ages=ages,
                                                              test_expressions = gene_expressions_test,
                                                              marker_genes=predictor_genes, min_age=2, max_age=23)

In [ ]:
predictions_df, weights_df = predict_test(train_expressions=train_expressions,
                                          train_ages=train_ages,
                                          test_expressions=test_expressions)

In [ ]:
save_weights_plot(test_predictions=predictions_df, test_prediction_weights=weights_df,
                  train_ages=train_ages,
                  folder="deconvolution/plots/weights/ref_2_23_markergene_2_23")

In [ ]:
predictions_df.to_csv("deconvolution/plots/weights/ref_2_23_markergene_2_23/test_predictions.csv",
                      index=False)
weights_df.to_csv("deconvolution/plots/weights/ref_2_23_markergene_2_23/test_prediction_weights.csv")

## Reference Panel 6-23, Marker Gene 6-23

In [ ]:
# predictor genes
predictor_genes_df = pd.read_csv("gene_plots/top_gene_6_23/markergene_6_23.csv", index_col=0)
predictor_genes = predictor_genes_df.index.tolist()

In [ ]:
train_ages, train_expressions, test_expressions = subset_data(train_expressions=gene_expressions,
                                                              train_ages=ages,
                                                              test_expressions = gene_expressions_test,
                                                              marker_genes=predictor_genes, min_age=6, max_age=23)

In [ ]:
predictions_df, weights_df = predict_test(train_expressions=train_expressions,
                                          train_ages=train_ages,
                                          test_expressions=test_expressions)

In [ ]:
save_weights_plot(test_predictions=predictions_df, test_prediction_weights=weights_df,
                  train_ages=train_ages,
                  folder="deconvolution/plots/weights/ref_6_23_markergene_6_23")

In [ ]:
predictions_df.to_csv("deconvolution/plots/weights/ref_6_23_markergene_6_23/test_predictions.csv",
                      index=False)
weights_df.to_csv("deconvolution/plots/weights/ref_6_23_markergene_6_23/test_prediction_weights.csv")

## Reference panel 2-23, Marker gene 6-23

In [ ]:
# predictor genes
predictor_genes_df = pd.read_csv("gene_plots/top_gene_6_23/markergene_6_23.csv", index_col=0)
predictor_genes = predictor_genes_df.index.tolist()

In [ ]:
train_ages, train_expressions, test_expressions = subset_data(train_expressions=gene_expressions,
                                                              train_ages=ages,
                                                              test_expressions = gene_expressions_test,
                                                              marker_genes=predictor_genes, min_age=2, max_age=23)

In [ ]:
predictions_df, weights_df = predict_test(train_expressions=train_expressions,
                                          train_ages=train_ages,
                                          test_expressions=test_expressions)

In [ ]:
save_weights_plot(test_predictions=predictions_df, test_prediction_weights=weights_df,
                  train_ages=train_ages,
                  folder="deconvolution/plots/weights/ref_2_23_markergene_6_23")

In [ ]:
predictions_df.to_csv("deconvolution/plots/weights/ref_2_23_markergene_6_23/test_predictions.csv",
                      index=False)
weights_df.to_csv("deconvolution/plots/weights/ref_2_23_markergene_6_23/test_prediction_weights.csv")

## Reference Panel 2-18, Marker Gene 6-18

In [ ]:
# predictor genes
predictor_genes_df = pd.read_csv("gene_plots/top_gene_6_18/markergene_6_18.csv", index_col=0)
predictor_genes = predictor_genes_df.index.tolist()

In [ ]:
train_ages, train_expressions, test_expressions = subset_data(train_expressions=gene_expressions,
                                                              train_ages=ages,
                                                              test_expressions = gene_expressions_test,
                                                              marker_genes=predictor_genes, min_age=2, max_age=18)

In [ ]:
predictions_df, weights_df = predict_test(train_expressions=train_expressions,
                                          train_ages=train_ages,
                                          test_expressions=test_expressions)

In [ ]:
save_weights_plot(test_predictions=predictions_df, test_prediction_weights=weights_df,
                  train_ages=train_ages,
                  folder="deconvolution/plots/weights/ref_2_18_markergene_6_18")

In [ ]:
predictions_df.to_csv("deconvolution/plots/weights/ref_2_18_markergene_6_18/test_predictions.csv",
                      index=False)
weights_df.to_csv("deconvolution/plots/weights/ref_2_18_markergene_6_18/test_prediction_weights.csv")

## Reference Panel 2-18, Marker Gene 2-18

In [ ]:
# predictor genes
predictor_genes_df = pd.read_csv("gene_plots/top_gene_2_18/markergene_2_18.csv", index_col=0)
predictor_genes = predictor_genes_df.index.tolist()

In [ ]:
train_ages, train_expressions, test_expressions = subset_data(train_expressions=gene_expressions,
                                                              train_ages=ages,
                                                              test_expressions = gene_expressions_test,
                                                              marker_genes=predictor_genes, min_age=2, max_age=18)

In [ ]:
predictions_df, weights_df = predict_test(train_expressions=train_expressions,
                                          train_ages=train_ages,
                                          test_expressions=test_expressions)

In [ ]:
save_weights_plot(test_predictions=predictions_df, test_prediction_weights=weights_df,
                  train_ages=train_ages,
                  folder="deconvolution/plots/weights/ref_2_18_markergene_2_18")

In [ ]:
predictions_df.to_csv("deconvolution/plots/weights/ref_2_18_markergene_2_18/test_predictions.csv",
                      index=False)
weights_df.to_csv("deconvolution/plots/weights/ref_2_18_markergene_2_18/test_prediction_weights.csv")

## Reference Panel 6-18, Marker Gene 6-18

In [ ]:
# predictor genes
predictor_genes_df = pd.read_csv("gene_plots/top_gene_6_18/markergene_6_18.csv", index_col=0)
predictor_genes = predictor_genes_df.index.tolist()

In [ ]:
train_ages, train_expressions, test_expressions = subset_data(train_expressions=gene_expressions,
                                                              train_ages=ages,
                                                              test_expressions = gene_expressions_test,
                                                              marker_genes=predictor_genes, min_age=6, max_age=18)

In [ ]:
predictions_df, weights_df = predict_test(train_expressions=train_expressions,
                                          train_ages=train_ages,
                                          test_expressions=test_expressions)

In [ ]:
save_weights_plot(test_predictions=predictions_df, test_prediction_weights=weights_df,
                  train_ages=train_ages,
                  folder="deconvolution/plots/weights/ref_6_18_markergene_6_18")

In [ ]:
predictions_df.to_csv("deconvolution/plots/weights/ref_6_18_markergene_6_18/test_predictions.csv",
                      index=False)
weights_df.to_csv("deconvolution/plots/weights/ref_6_18_markergene_6_18/test_prediction_weights.csv")